# Import packages and data loading

In [7]:
import time
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src import (
    load_instance,
    build_model,
    solve_model,
    solve_and_summarize_all,
    extract_assignment,
    print_summary,
    get_model_stats,
    display_solution,
    break_ties_randomly,
    compute_policy_metrics
)

import json
import pandas as pd

in this section we work with models that use Hungarian policy to solve problems with possible ties. bacause of this, we don't need preprocessed datasets and the simple datasets are enough.

In [8]:
data_small = json.load(open(r'../data/generated/instance_small.json', 'r', encoding='utf-8'))
print("Loaded small instance:\n   n={}, m={}".format(data_small['n'], data_small['m']))

data_medium = json.load(open(r'../data/generated/instance_medium.json', 'r', encoding='utf-8'))
# data_small = load_instance(filename='../data/generated/instance_small.json')
data_medium = load_instance(filename='../data/generated/instance_medium.json')
print("Loaded medium instance:\n   n={}, m={}".format(data_medium['n'], data_medium['m']))

Loaded small instance:
   n=10, m=5
Loaded medium instance:
   n=50, m=20


# Solving Models

first we run the Pyomo models for Chilean policy:
- `SO-C-NW-CUT`
- `SO-C-NW-BIN-CUT`

these models makes it possible for ties to violate quota constraint to accept applicants with same score.

In [9]:
section_four_chilean_formulations = ["SO-C-NW-CUT", "SO-C-NW-BIN-CUT"]

results_small = solve_and_summarize_all(data_small, solver_name='cplex', tee=False, formulations=section_four_chilean_formulations)


print("\nSUMMARY TABLE")
print("="*100)

df_summary = pd.DataFrame([
    {
        'Formulation': r['model'],
        'Status': r['status'].upper(),
        'Model Obj': f"{r['model_objective']:.2f}" if r['model_objective'] is not None else "N/A",
        'Rank Obj': f"{r['rank_objective']}" if r['rank_objective'] is not None else "N/A",
        'Matched': r['students_assigned'],
        'Avg Rank': f"{r['rank_objective']/max(1, r['students_assigned']):.2f}" if r['rank_objective'] is not None and r['students_assigned'] > 0 else "N/A"
    }
    for r in results_small
])

print(df_summary.to_string(index=False))

KeyboardInterrupt: 

# Policy comparison

in this section we use the metrics below to compare the models `SO-H-NW-BIN-CUT` (for Hungarian policy), `SO-C-NW-BIN-CUT` (for Chilean policy), and `SO-NW-BIN-CUT` (for Irish policy):

- size of matching
- average rank
- average cutoffs
- number of rejections

the details for computing these metrics is in `utility.py` function `compute_policy_metrics()`. in this table, beside the original models, each time we once compute the model with flipped objective function (min to max or max to min) to find the Student-Pessimal, the worst stable matching for students.
since `pyomo` allows to change the objective senses after building, we don't change the original models and we apply this change here to find C-opt solutions.

In [ ]:
def solve_with_opt(data, formulation, opt_type, solver='cplex'):
    """
    opt_type: 'A-opt' or 'C-opt'
    """
    import pyomo.environ as pyo
    model = build_model(data, formulation=formulation)
    
    if formulation in ["SO-H-NW-BIN-CUT", "SO-C-NW-BIN-CUT"]:
        if opt_type == 'A-opt':
            model.Objective.sense = pyo.maximize
        else:
            model.Objective.sense = pyo.minimize
    elif formulation == "SO-NW-BIN-CUT":
        if opt_type == 'A-opt':
            model.Objective.sense = pyo.minimize
        else:
            model.Objective.sense = pyo.maximize
            
    solver = pyo.SolverFactory(solver)
    result = solver.solve(model, tee=False)
    return model

In [ ]:
def solve_with_opt(data, formulation, opt_type, solver='cplex'):
    """
    opt_type: 'A-opt' or 'C-opt'
    Returns: model (and optionally status)
    """
    import pyomo.environ as pyo
    model = build_model(data, formulation=formulation)
    
    if formulation in ["SO-H-NW-BIN-CUT", "SO-C-NW-BIN-CUT"]:
        if opt_type == 'A-opt':
            model.Objective.sense = pyo.maximize
        else:
            model.Objective.sense = pyo.minimize
    elif formulation == "SO-NW-BIN-CUT":
        if opt_type == 'A-opt':
            model.Objective.sense = pyo.minimize
        else:
            model.Objective.sense = pyo.maximize

    solver_obj = pyo.SolverFactory(solver)
    result = solver_obj.solve(model, tee=False)
    
    return model

In [ ]:
policies = {
    'Hungarian': 'SO-H-NW-BIN-CUT',
    'Chilean': 'SO-C-NW-BIN-CUT',
    'Irish': 'SO-NW-BIN-CUT'
}

rows = []

data_irish = break_ties_randomly(data_medium)

for policy_name, formulation in policies.items():
    for opt in ['A-opt', 'C-opt']:
        if policy_name == 'Irish':
            # For Irish, we need fresh random tie-breaking each time
            data_to_use = data_irish
        else:
            data_to_use = data_medium
        
        # print(f"Solving {policy_name} {opt}...")
        # model = solve_with_opt(data_to_use, formulation, opt)
        
        metrics = compute_policy_metrics(model, data_to_use)
        rows.append({
            'Policy': policy_name,
            'Opt': opt,
            'Size': metrics['size'],
            'Avg Rank': round(metrics['avg_rank'], 4),
            'Avg Cutoffs': round(metrics['avg_cutoffs'], 4),
            '# Rejections': metrics['rejections']
        })

df = pd.DataFrame(rows)
print("\nTable 4: Comparison of student-optimal (A-opt) and student-pessimal (C-opt) stable matchings")
print(df.to_string(index=False))

Solving Hungarian A-opt...
  Solver status for SO-H-NW-BIN-CUT A-opt: optimal
  -> size=50, avg_rank=0.6800, avg_cutoffs=20.9000, rejections=0
Solving Hungarian C-opt...
  Solver status for SO-H-NW-BIN-CUT C-opt: optimal
  -> size=49, avg_rank=5.1224, avg_cutoffs=41.5500, rejections=1
Solving Chilean A-opt...
  Solver status for SO-C-NW-BIN-CUT A-opt: optimal
  -> size=50, avg_rank=0.6800, avg_cutoffs=20.9000, rejections=0
Solving Chilean C-opt...
  Solver status for SO-C-NW-BIN-CUT C-opt: optimal
  -> size=0, avg_rank=0.0000, avg_cutoffs=0.0000, rejections=50
Solving Irish A-opt...
  Solver status for SO-NW-BIN-CUT A-opt: optimal
  -> size=50, avg_rank=0.6800, avg_cutoffs=20.9000, rejections=0
Solving Irish C-opt...
  Solver status for SO-NW-BIN-CUT C-opt: optimal
  -> size=50, avg_rank=0.6800, avg_cutoffs=20.9000, rejections=0

Table 4: Comparison of student-optimal (A-opt) and student-pessimal (C-opt) stable matchings
   Policy   Opt  Size  Avg Rank  Avg Cutoffs  # Rejections
Hungar